# Assignment 3 - Wall-Following Control Study

Had Codex turn the original .py file into a jupyter notebook.

## Setup

Run this notebook in the same folder as wf_sim.py. Rerun the import cell after any edits to the simulator.

In [ ]:
import importlib
import numpy as np
import matplotlib.pyplot as plt

import wf_sim
importlib.reload(wf_sim)

SimConfig = wf_sim.SimConfig
PController = wf_sim.PController
PIDController = wf_sim.PIDController
run_sim = wf_sim.run_sim
metrics = wf_sim.metrics
plot_runs = wf_sim.plot_runs

def metric_row(label, log):
    return {'case': label, **metrics(log)}

def print_metric_table(rows):
    print('{:>5}  {:>8}  {:>13}  {:>8}  {:>7}'.format('Kp', 'RMS (m)', 'Overshoot (m)', 'Effort', 'Misses'))
    for row in rows:
        print('{:5.1f}  {:8.3f}  {:13.3f}  {:8.2f}  {:6.1f}%'.format(row['Kp'], row['rms_error'], row['max_overshoot'], row['control_effort'], row['deadline_miss_pct']))


## 1. Orientation

In [ ]:
# TODO: select a provided P gain, run a baseline, and inspect its trajectory and metrics.
# cfg = SimConfig(...)
# baseline_log = run_sim(cfg, PController(...))
# pd.DataFrame([metric_row('baseline P', baseline_log)])


## 2. Proportional-control sweep

In [ ]:
# Question 2: proportional-control characterization.
kp_values = [0.2, 0.5, 1.5, 4.0, 8.0]
p_runs = []
p_rows = []

for kp in kp_values:
    log = run_sim(SimConfig(), PController(kp))
    p_runs.append((f'Kp = {kp}', log))
    p_rows.append({'Kp': kp, **metrics(log)})

print_metric_table(p_rows)

fig, ax = plt.subplots(figsize=(9, 4.8))
for label, log in p_runs:
    ax.plot(log['t'], log['d_star'] - log['y'], label=label, linewidth=1.35)
ax.axhline(0.0, color='black', linewidth=0.8, alpha=0.6)
ax.set(title='Question 2: Proportional-control gain sweep',
       xlabel='Time (s)', ylabel='Tracking error d* - y (m)')
ax.grid(alpha=0.3)
ax.legend(ncol=3)
fig.tight_layout()
fig.savefig('question2_p_sweep.png', dpi=160)
plt.show()


### Question 2 results

| Kp | RMS error (m) | Max overshoot (m) | Control effort | Deadline misses |
| ---: | ---: | ---: | ---: | ---: |
| 0.2 | 0.739 | 1.081 | 5.25 | 0.0% |
| 0.5 | 0.769 | 1.190 | 13.48 | 0.0% |
| 1.5 | 0.716 | 1.231 | 34.58 | 0.0% |
| 4.0 | 0.711 | 1.017 | 59.46 | 0.0% |
| 8.0 | 0.701 | 1.000 | 60.07 | 0.0% |

the low gain regine is sluggish around k = 2. The robot will take a long time to reverse its motion. Around kp = 0.5 its more responsive, but still slow. Though it avoids aggressive steering. By kp=1 oscillation is noticable and strongly oscillates. kp 4 and 8 are similar because of the omega limit. The controller is bottlennecked by the turn rate. No gain can make the curve flat because steering is error based. the robot keeps moving as its heading changes, so it carries lateral motion past the target distance before the error can reverse the turn. The shortest correction time will always butt up against the error.

## 3. PID implementation, tuning, and ablation

In [ ]:
# Question 3: tuned PID step response and P/PD/PID ablation.
step_cfg = SimConfig(setpoint_step_time=20.0, setpoint_step_value=1.5)
ablation_rows = []
ablation_runs = []

# Tuned full-PID gains: Kp = 0.05, Ki = 0.001, Kd = 0.40.
controllers = [
    ('P-only', PController(0.05)),
    ('PD', PIDController(0.05, 0.0, 0.40)),
    ('Full PID', PIDController(0.05, 0.001, 0.40)),
]

for label, controller in controllers:
    log = run_sim(step_cfg, controller)
    ablation_runs.append((label, log))
    ablation_rows.append({'case': label, **metrics(log)})

print('{:>9}  {:>8}  {:>10}  {:>8}  {:>7}'.format('Controller', 'RMS (m)', 'Overshoot', 'Effort', 'Misses'))
for row in ablation_rows:
    print('{:>9}  {:8.3f}  {:10.3f}  {:8.2f}  {:6.1f}%'.format(row['case'], row['rms_error'], row['max_overshoot'], row['control_effort'], row['deadline_miss_pct']))

fig, (ax_path, ax_error) = plt.subplots(2, 1, figsize=(9, 7), sharex=False)
for label, log in ablation_runs:
    ax_path.plot(log['x'], log['y'], label=label)
    ax_error.plot(log['t'], log['d_star'] - log['y'], label=label)
ax_path.axhline(0.0, color='black', linewidth=2, label='wall')
ax_path.axhline(1.0, color='gray', linestyle='--', label='initial target')
ax_path.axhline(1.5, color='gray', linestyle=':', label='step target')
ax_path.set(title='Question 3: PID step response and ablation', ylabel='Distance to wall y (m)')
ax_path.grid(alpha=0.3)
ax_path.legend(ncol=2)
ax_error.axhline(0.0, color='black', linewidth=0.8)
ax_error.axvline(20.0, color='gray', linestyle='--', label='setpoint step')
ax_error.set(xlabel='Time (s)', ylabel='Tracking error d* - y (m)')
ax_error.grid(alpha=0.3)
ax_error.legend()
fig.tight_layout()
fig.savefig('question3_pid_ablation.png', dpi=160)
plt.show()


### Question 3 results

The tuned PID controller uses PID 0.05, 0.001 and 0.4 respectively. The RMS metric reported by the starter function includes the initial startup transient so it is not the same as a settled error measurement. Using the settled intervals the full PID RMS errors are 0.046m and 0.038 respectively.

| Controller | RMS error (m) | Max overshoot (m) | Control effort | Deadline misses |
| --- | ---: | ---: | ---: | ---: |
| P-only | 0.964 | 1.529 | 1.60 | 0.0% |
| PD | 0.249 | 1.000 | 7.40 | 0.0% |
| Full PID | 0.264 | 1.000 | 7.40 | 0.0% |

P only has the least control effort but its turn rate correction is too weak and has no damping. This cuases it to retain a large tracking error. Adding D improves damping. It opposes rapidly changing error and prevents large oscillations of a P only response. Adding I removes bias over time. It has the smallest improvement of all.

## 4. Timing: sampling, latency, and jitter


In [ ]:
# Question 4: timing study using the tuned PID from Question 3.
tuned_gains = (0.05, 0.001, 0.40)

# 4a: sample-rate sweep. The final two points extend the suggested range
# so that the breakdown point is observable.
sampling_periods = [0.01, 0.05, 0.20, 0.50, 1.00, 3.00]
sampling_rows = []
for dt in sampling_periods:
    cfg = SimConfig(setpoint_step_time=20.0, setpoint_step_value=1.5,
                    dt_nominal=dt, deadline=dt)
    log = run_sim(cfg, PIDController(*tuned_gains))
    sampling_rows.append({'dt': dt, 'Hz': 1.0 / dt, **metrics(log)})

# 4b: latency sweep at the nominal 20 Hz loop rate.
latency_values = [0, 5, 10, 20, 40, 80]
latency_rows = []
for steps in latency_values:
    cfg = SimConfig(setpoint_step_time=20.0, setpoint_step_value=1.5,
                    latency_steps=steps)
    log = run_sim(cfg, PIDController(*tuned_gains))
    latency_rows.append({'steps': steps, 'delay_s': steps * cfg.dt_nominal, **metrics(log)})

# 4c: jitter sweep. A 40 ms nominal loop and 50 ms deadline leave 10 ms slack.
jitter_values = [0.0, 0.005, 0.010, 0.020, 0.030, 0.040, 0.050]
jitter_rows = []
for jitter in jitter_values:
    cfg = SimConfig(setpoint_step_time=20.0, setpoint_step_value=1.5,
                    dt_nominal=0.040, deadline=0.050, jitter_std=jitter)
    log = run_sim(cfg, PIDController(*tuned_gains))
    jitter_rows.append({'jitter_std_s': jitter, **metrics(log)})

fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
axes[0].plot([r['Hz'] for r in sampling_rows], [r['rms_error'] for r in sampling_rows], 'o-', label='RMS error')
axes[0].set_xscale('log')
axes[0].set(title='Sampling rate', xlabel='Rate (Hz, log scale)', ylabel='RMS error (m)')
axes[0].grid(alpha=0.3)
axes[1].plot([r['delay_s'] for r in latency_rows], [r['rms_error'] for r in latency_rows], 'o-', color='tab:orange')
axes[1].set(title='Measurement latency', xlabel='Delay (s)', ylabel='RMS error (m)')
axes[1].grid(alpha=0.3)
axes[2].plot([1000 * r['jitter_std_s'] for r in jitter_rows], [r['deadline_miss_pct'] for r in jitter_rows], 'o-', color='tab:red')
axes[2].axhline(5.0, color='black', linestyle='--', label='5% criterion')
axes[2].set(title='Jitter and deadline misses', xlabel='Jitter std. dev. (ms)', ylabel='Deadline misses (%)')
axes[2].grid(alpha=0.3)
axes[2].legend()
fig.tight_layout()
fig.savefig('question4_timing_study.png', dpi=160)
plt.show()


### Question 4 results

#### (a) Sampling rate

| Rate (Hz) | RMS (m) | Overshoot (m) | Effort | Deadline misses |
| ---: | ---: | ---: | ---: | ---: |
| 100.0 | 0.323 | 1.000 | 32.20 | 0.0% |
| 20.0 | 0.264 | 1.000 | 7.40 | 0.0% |
| 5.0 | 0.198 | 1.000 | 2.00 | 0.0% |
| 2.0 | 0.199 | 1.000 | 1.05 | 0.0% |
| 1.0 | 0.198 | 1.000 | 0.94 | 0.0% |
| 0.33 | 0.568 | 1.000 | 2.47 | 0.0% |


Control remains usable through 2hz. It becomes unusable around a third of a hz. Here, RMS rises to 0.568. At this rate, the controller holds each steering decision while the robot moves substantially. Correction arrives after the state has changed. The 100hz run has higher effor because this unfiltered derivative term reacts more often to sensor noise.

#### (b) Latency

| Delay (steps / s) | RMS (m) | Overshoot (m) | Effort | Deadline misses |
| --- | ---: | ---: | ---: | ---: |
| 0 / 0.00 | 0.264 | 1.000 | 7.40 | 0.0% |
| 5 / 0.25 | 0.258 | 1.000 | 7.38 | 0.0% |
| 10 / 0.50 | 0.249 | 1.000 | 7.35 | 0.0% |
| 20 / 1.00 | 0.255 | 1.000 | 7.30 | 0.0% |
| 40 / 2.00 | 0.246 | 1.000 | 7.13 | 0.0% |
| 80 / 4.00 | 0.439 | 1.000 | 6.99 | 0.0% |


small delays do not worsen this particular noisy, conservatively tuned sim. They can slightly smooth the measured signal. This is not a stability benefit to rely on. At 4s of delay. RMS degrades because the controller is correcting an error from a state the robot left some time ago. Delay generally removes phase margin. A correction can push in the wrong direction after the true error has already been rectified

#### (c) Jitter

For this test the loop niminal period is 40ms and deadline is 50ms. this leaves 10ms of timing slack. We can classify jitter as problematic when more than 5% of loops miss the deadline or when RMS increase by more than 20% compared with no jitter run. The 5% deadline threshold is exceeded at a jitter stdev of 10ms. 10ms and above are problematic

| Jitter std. dev. (ms) | RMS (m) | Effort | Deadline misses |
| ---: | ---: | ---: | ---: |
| 0 | 0.274 | 8.92 | 0.0% |
| 5 | 0.271 | 9.02 | 1.7% |
| 10 | 0.252 | 9.05 | 15.4% |
| 20 | 0.291 | 8.78 | 31.8% |
| 30 | 0.249 | 7.89 | 37.0% |
| 40 | 0.275 | 6.97 | 41.3% |
| 50 | 0.330 | 6.16 | 43.2% |


Motor commands become less trustworthy when their timining is unpredictable relative to how quickly the robot came move. route updates and displats are soft real time because a late updated is degraded but usually usable. Motor control is hard real time because repeated missed control deadlines can destabilize motion. an emergency stop is a hard real time and endangered most by jitter because a late stop can result in a safety failure if its not achieved in time.

## 5. Open-world safety

In [ ]:
# Question 5: compare the same obstacle course with and without safety.
importlib.reload(wf_sim)
obstacle_x = 5.0
controller_gains = (0.05, 0.001, 0.40)

unsafe_cfg = wf_sim.SimConfig(T=20.0, obstacle_x=obstacle_x, safety_override_enabled=False)
safe_cfg = wf_sim.SimConfig(T=20.0, obstacle_x=obstacle_x, safety_override_enabled=True)
unsafe_log = wf_sim.run_sim(unsafe_cfg, wf_sim.PIDController(*controller_gains))
safe_log = wf_sim.run_sim(safe_cfg, wf_sim.PIDController(*controller_gains))

print('No override: final x = {:.2f} m; obstacle at x = {:.2f} m'.format(unsafe_log['x'][-1], obstacle_x))
print('Safety override: final x = {:.2f} m; activations = {}'.format(safe_log['x'][-1], safe_log['overridden'].sum()))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(unsafe_log['x'], unsafe_log['y'], label='No override', linewidth=2)
ax.plot(safe_log['x'], safe_log['y'], label='Safety override', linewidth=2)
ax.axhline(0.0, color='black', linewidth=2, label='wall')
ax.axvline(obstacle_x, color='tab:red', linestyle='--', linewidth=2, label='obstacle')
ax.axvline(obstacle_x - safe_cfg.SAFE_DIST, color='tab:orange', linestyle=':', label='0.6 m safety boundary')
ax.set(title='Question 5: obstacle response', xlabel='x position (m)', ylabel='distance to wall y (m)')
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig('question5_safety_override.png', dpi=160)
plt.show()


### Question 5 results



With the obstacle at x = 5m and the safety override idsabled, the robot reaches x=9.93m, so it passes through the blocked area. The wall distance feedback controller can be working correctly while knowing nothing about an obstacle directly ahead. with the override enabled, the robot stops at x = 4.41m, approx. the 0.6m safety boundary before the obstacle and the log records 222 override activations.

feedback is needed because it keeps correcting distance from the wall in response to senses error, but it is not sufficident for open world safety because the feedback signal contains only wall distance not all hazards. the obstacle is outside the wall follow controllers model so it cant do anything about it even when correct. Safety override checks for obstacles independently and replaces the nominal command with a safe fallback, which is 0 forward speed and turning from the wall. This is a reactive architecture. It maps immediate sensor condition directly to a protective action without any sort of long horizon planning.


## 6. Bounded-rationality autotuner

In [ ]:
# Question 6: bounded grid search. autotune is implemented in wf_sim.py.
importlib.reload(wf_sim)
tuning_cfg = wf_sim.SimConfig(setpoint_step_time=20.0, setpoint_step_value=1.5)

# Cost = RMS tracking error + 0.01 * control effort.
small_result = wf_sim.autotune(tuning_cfg, budget=10)
large_result = wf_sim.autotune(tuning_cfg, budget=100)

for label, result in [('Small budget', small_result), ('Large budget', large_result)]:
    gains, result_metrics, cost, evaluated = result
    print('{}: {} candidates, gains Kp={:.2f}, Ki={:.3f}, Kd={:.2f}, cost={:.3f}'.format(label, evaluated, *gains, cost))
    print('  RMS={:.3f} m, overshoot={:.3f} m, effort={:.2f}, misses={:.1f}%'.format(result_metrics['rms_error'], result_metrics['max_overshoot'], result_metrics['control_effort'], result_metrics['deadline_miss_pct']))

fig, ax = plt.subplots(figsize=(5.8, 4))
ax.bar(['Small budget (10 candidates)', 'Large budget (100 candidates)'],
       [small_result[2], large_result[2]], color=['tab:orange', 'tab:green'])
ax.set(title='Question 6: bounded-search cost', ylabel='RMS + 0.01 * effort')
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig('question6_autotune_budget.png', dpi=160)
plt.show()


### Question 6 results

The autotuner evaluates a bounded grid of 100 gain triples. 5 Kp values, 4 Ki values, and 5 Kd values. For every candidate, it runs the simulator and minimizes the stated cost, RMS tracking error + 0.01 times control effort. The budget check occurs before each simulation, so the small run cannot exceed 10 evaluations and the large run cannot exceed 100.

| Search budget | Candidates evaluated | Best gains (Kp, Ki, Kd) | RMS (m) | Overshoot (m) | Effort | Cost |
| ---: | ---: | --- | ---: | ---: | ---: | ---: |
| 10 | 10 | (0.05, 0.000, 0.40) | 0.249 | 1.000 | 7.39 | 0.323 |
| 100 | 100 | (0.20, 0.000, 0.40) | 0.165 | 1.000 | 7.72 | 0.242 |

The small budget search returns a worse controller because it sees only the first 10 candidates in the feasible gain grid. The large budget one evaluates every feasible candidate and discovers a setting with lower tracking error for only a little more control effor. This is bounded rationality. The agent is not selecting the best action it could find with unlimited compute and complete search, but the best it can within budget. 



Feedback/error correction: The controller repeatedly measures e = d* - d and changes omega from that error. In Q3, adding the D term reduced the large P only oscillations because it reacted to how quickly the error was changing, not just its current size.

Soft versus hard real time: Q4 showed that 10 ms jitter caused about 15 percent of loops to miss a 50 ms deadline. A delayed display or route update is soft real time because it is still useful when late. Motor control and especially an emergency stop are hard real time because a late action can cause unsafe motion.

Closed versus open world: The wall following controller assumes the only important feature is the wall distance, so without the Q5 override it passed the obstacle at x = 5m. The reactive safety layer uses a separate front obstacle signal to stop before the hazard, making the system safer in an open world environment 